In [6]:

import pandas as pd
import re


invoices = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned_brief_c\invoices.csv")
payments = pd.read_csv(r"C:\Users\Diksha\Documents\cleaned_brief_c\gateway_payments.csv")


def normalize_invoice_id(value):
    if pd.isna(value):
        return None
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

invoices["match_id"] = invoices["invoice_id"].apply(
    normalize_invoice_id
)

payments["match_id"] = payments["invoice_ref"].apply(
    normalize_invoice_id
)


invoices["amount_rs"] = pd.to_numeric(
    invoices["amount_rs"], errors="coerce"
)

payments["amount_rs"] = pd.to_numeric(
    payments["amount_rs"], errors="coerce"
)


payments = payments.drop_duplicates(
    subset=["payment_id"], keep="first"
)

valid_payments = payments[
    payments["status"].astype(str).str.strip().str.lower()
    == "captured"
].copy()


payment_totals = (
    valid_payments
    .dropna(subset=["match_id"])
    .groupby("match_id", as_index=False)
    .agg(
        captured_amount_rs=("amount_rs", "sum")
    )
)


reconciliation = invoices.merge(
    payment_totals,
    on="match_id",
    how="left"
)

reconciliation["captured_amount_rs"] = (
    reconciliation["captured_amount_rs"].fillna(0)
)

# 8. Calculate outstanding balances
reconciliation["outstanding_rs"] = (
    reconciliation["amount_rs"]
    - reconciliation["captured_amount_rs"]
).clip(lower=0)


def classify_invoice(row):
    invoice_amount = row["amount_rs"]
    paid_amount = row["captured_amount_rs"]

    if pd.isna(invoice_amount):
        return "Invalid invoice amount"
    elif paid_amount == 0:
        return "Unpaid"
    elif paid_amount < invoice_amount:
        return "Partially paid"
    elif paid_amount == invoice_amount:
        return "Paid in full"
    else:
        return "Overpaid"

reconciliation["invoice_status"] = reconciliation.apply(
    classify_invoice, axis=1
)


unpaid_count = (
    reconciliation["invoice_status"] == "Unpaid"
).sum()

partial_count = (
    reconciliation["invoice_status"] == "Partially paid"
).sum()

total_outstanding_count = (
    reconciliation["invoice_status"].isin(
        ["Unpaid", "Partially paid"]
    )
).sum()

total_outstanding_amount = (
    reconciliation.loc[
        reconciliation["invoice_status"].isin(
            ["Unpaid", "Partially paid"]
        ),
        "outstanding_rs"
    ].sum()
)

print("Unpaid invoices:", unpaid_count)
print("Partially paid invoices:", partial_count)
print("Total invoices with outstanding balances:",
      total_outstanding_count)
print("Total outstanding amount (Rs):",
      round(total_outstanding_amount, 2))


Unpaid invoices: 21820
Partially paid invoices: 49
Total invoices with outstanding balances: 21869
Total outstanding amount (Rs): 34878773.0


In [5]:



def normalize_invoice_id(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())

invoices["invoice_key"] = invoices["invoice_id"].apply(normalize_invoice_id)
payments["invoice_key"] = payments["invoice_ref"].apply(normalize_invoice_id)


captured = payments[
    payments["status"].str.lower().eq("captured")
].copy()


unmatched = captured[
    ~captured["invoice_key"].isin(invoices["invoice_key"])
]


print("Unmatched captured payment rows:", len(unmatched))
print("Unique unmatched payment IDs:", unmatched["payment_id"].nunique())
print("Total amount in unmatched rows (Rs):", unmatched["amount_rs"].sum())

Unmatched captured payment rows: 4588
Unique unmatched payment IDs: 4588
Total amount in unmatched rows (Rs): 7269408.0


In [ ]:
# Left join: keep all gateway payments and match invoice details
gateway_reconciliation = payments.merge(
    invoices,
    on="invoice_key",
    how="left",
    suffixes=("_payment", "_invoice"),
    indicator=True
)

# Print the complete joined dataset
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(gateway_reconciliation)